# 01 - The data: corpora, vectors, and where they live

**What this notebook is for.** Everything downstream (geometry, probes) rests on text corpora
and the emotion vectors extracted from them. This notebook is the catalog: what each dataset
is, who generated it, its quality control, and where the published copy lives.

**Key concepts.**
- *Emotion story corpus*: short texts written to evoke one emotion; the model reads them and
  we average its internal activations per emotion to get one *emotion vector* each.
- *Leakage*: a generated text naming its target emotion. Low leakage means vectors encode the
  concept, not the word.
- *HF*: Hugging Face, where datasets are published (private to the team for now).

**Index.**
1. Corpus catalog
2. Leakage quality control
3. Vector sets and their homes

## 1. Corpus catalog

In [1]:
# this cell tabulates every corpus: size, generator, role (all counts computed from the files)
import json
from pathlib import Path

import plotly.graph_objects as go

from emotion_vectors.artifacts import ROUTES, fetch  # local results/ first, HF otherwise
from emotion_vectors.corpus import load_emotions_data

ROOT = Path("..")


def corpus_stats(path):
    """(n emotion groups, n texts) for a grouped jsonl whose stories are plain strings."""
    rows = [json.loads(line) for line in open(path)]
    return len(rows), sum(len(row["stories"]) for row in rows)


# the published reference corpus lives on HF, not under results/: count it from the dataset rows
published = load_emotions_data("snae/emotion_stories_gemma_4_4B", "train")
published_stats = (len(published), sum(len(stories) for stories in published.values()))

# the Q3 combined-stories corpus groups by emotion TRIPLE (each story is a dict with a
# "text" field), so it gets its own counter: (n distinct emotions, n story texts)
combined_rows = [json.loads(line) for line in open(fetch("combined_stories/stories_grouped.jsonl"))]
combined_stats = (
    len({emotion for row in combined_rows for emotion in row["emotions"]}),
    sum(len(row["stories"]) for row in combined_rows),
)

CATALOG = [
    (
        "published stories",
        "snae/emotion_stories_gemma_4_4B",
        "gemma-4-4B (reference authors)",
        "probe extraction, both models",
        *published_stats,
    ),
    (
        "self stories (n=256/emotion)",
        "results/self_stories_it/dialogues_grouped.jsonl",
        "gemma-4-31b-it (ours)",
        "scale test E6; self-generated lineage E10/E11",
        *corpus_stats(fetch("self_stories_it/dialogues_grouped.jsonl")),
    ),
    (
        "dialogues (base)",
        "results/dialogue_stories/dialogues_grouped.jsonl",
        "gemma-4-31b base (ours)",
        "dialogue-transfer E3",
        *corpus_stats(fetch("dialogue_stories/dialogues_grouped.jsonl")),
    ),
    (
        "dialogues (instruct)",
        "results/dialogue_stories_it/dialogues_grouped.jsonl",
        "gemma-4-31b-it (ours)",
        "E5 pilot",
        *corpus_stats(fetch("dialogue_stories_it/dialogues_grouped.jsonl")),
    ),
    (
        "neutral transcripts",
        "results/neutral_transcripts_it/dialogues_grouped.jsonl",
        "gemma-4-31b-it (ours)",
        "confound projection E7",
        *corpus_stats(fetch("neutral_transcripts_it/dialogues_grouped.jsonl")),
    ),
    (
        "combined stories (Q3)",
        "results/combined_stories/stories_grouped.jsonl",
        "gemma-4-31b-it (ours)",
        "Q3 transition trajectories, 173 emotion triples",
        *combined_stats,
    ),
    (
        "DeepSeek, fixed prompt",
        "results/openrouter_stories/stories_grouped.jsonl",
        "deepseek-v4-pro (ours)",
        "external-generator lineage E11",
        *corpus_stats(fetch("openrouter_stories/stories_grouped.jsonl")),
    ),
]
try:
    diverse_path = fetch("openrouter_stories_diverse/stories_grouped.jsonl")
    CATALOG.append(
        (
            "DeepSeek, diverse prompts",
            "results/openrouter_stories_diverse/stories_grouped.jsonl",
            "deepseek-v4-pro (ours)",
            "diverse-prompt scale arm E12",
            *corpus_stats(diverse_path),
        )
    )
except Exception as err:  # corpus only on HF and not fetchable here: say where it lives
    diverse_path = None
    print(
        f"diverse DeepSeek corpus not fetchable here ({type(err).__name__}); "
        f"it lives on HF at {ROUTES['openrouter_stories_diverse']}"
    )
fig = go.Figure(
    go.Table(
        header=dict(
            values=["corpus", "location", "generator", "role", "emotions", "texts"], align="left"
        ),
        cells=dict(values=list(zip(*CATALOG)), align="left", height=26),
    )
)
fig.update_layout(
    title="Every corpus in the project, its generator (gemma-4-4B reference, gemma-4-31b, gemma-4-31b-it, deepseek-v4-pro), and its role",
    height=420,
    margin=dict(t=50, b=10),
)
fig.show()


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


<details><summary><b>How to read this table</b></summary>

One row per corpus; the emotions and texts columns are computed in the cell above from the file in the location column (the published corpus from its Hugging Face rows, everything else from the grouped jsonl under `results/`, fetched from HF when absent locally). The combined stories (Q3) row is the one exception in shape: its file groups stories by emotion triple rather than by single emotion, so its emotions column counts distinct emotions appearing across the 173 triples.

</details>

## 2. Leakage quality control

In [2]:
# this cell measures emotion-word leakage per corpus (counted from the files) and plots the comparison
from emotion_vectors.scoring import EMOTION_STEMS, leakage


def leakage_counts(emotions_data):
    """(leaked, total) over the emotions covered by EMOTION_STEMS, mirroring scoring.leakage
    for corpora already loaded as {emotion: [story, ...]} dicts."""
    leaked = total = 0
    for emotion, stories in emotions_data.items():
        stems = EMOTION_STEMS.get(emotion)
        if not stems:
            continue
        for story in stories:
            total += 1
            leaked += any(stem in story.lower() for stem in stems)
    return leaked, total


BARS = [
    ("published stories (4B)", *leakage_counts(published)),
    ("dialogues, base model", *leakage(fetch("dialogue_stories/dialogues_grouped.jsonl"))),
    ("dialogues, instruct", *leakage(fetch("dialogue_stories_it/dialogues_grouped.jsonl"))),
    ("self stories, instruct", *leakage(fetch("self_stories_it/dialogues_grouped.jsonl"))),
    ("DeepSeek, fixed prompt", *leakage(fetch("openrouter_stories/stories_grouped.jsonl"))),
]
if diverse_path is not None:
    BARS.append(("DeepSeek, diverse prompts", *leakage(diverse_path)))
for name, leaked, total in BARS:
    print(f"{name}: {leaked}/{total} texts name their emotion ({leaked / max(total, 1):.1%})")
names = [row[0] for row in BARS]
values = [row[1] / max(row[2], 1) for row in BARS]
fig = go.Figure(
    go.Bar(
        x=names,
        y=[v * 100 for v in values],
        text=[f"{v:.1%}" for v in values],
        textposition="outside",
    )
)
fig.add_annotation(
    text="instruction-tuned generation respects 'do not name the emotion'; the base model does not",
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.13,
    showarrow=False,
)
fig.update_layout(
    title="Emotion-word leakage by corpus: gemma-4-4B, gemma-4-31b base, gemma-4-31b-it, deepseek-v4-pro (lower is better)",
    yaxis_title="texts naming their emotion (%)",
    height=420,
)
fig.show()


published stories (4B): 4/108 texts name their emotion (3.7%)
dialogues, base model: 84/192 texts name their emotion (43.8%)
dialogues, instruct: 0/192 texts name their emotion (0.0%)
self stories, instruct: 42/3072 texts name their emotion (1.4%)
DeepSeek, fixed prompt: 26/3070 texts name their emotion (0.8%)
DeepSeek, diverse prompts: 123/12262 texts name their emotion (1.0%)


<details><summary><b>How to read this figure</b></summary>

Each bar is one corpus; height is the share of texts that name their target emotion despite instructions not to, counted in the cell above over the 12 emotions with leakage stems (`emotion_vectors.scoring.EMOTION_STEMS`). The published corpus sets the reference level of a few percent. The base-model dialogue bar sits an order of magnitude above it, which is why base-arm dialogue probes carry a lexical confound (handled in the probe notebook).

</details>

## 3. Vector sets and their homes

Each corpus above was run through the extraction pipeline (`src/emotion_vectors/extraction.py`)
to produce per-story activation shards and per-emotion mean vectors at 20 layers. Published
sets, all private team datasets on Hugging Face under `abotresol/`:

| dataset | contents |
|---|---|
| `emotion-vectors-gemma-4-31b` | base-model vectors, published stories |
| `emotion-vectors-gemma-4-31b-it` | instruct-model vectors, published stories |
| `emotion-dialogue-vectors-gemma-4-31b(-it)` | dialogue-derived vectors, both arms |
| `emotion-selfstory-vectors-gemma-4-31b-it` | self-story vectors (scale corpus) |
| `neutral-vectors-gemma-4-31b-it` | neutral-transcript vectors (projection) |
| `emotion-dialogues-...`, `emotion-stories-...`, `neutral-transcripts-...` | the raw corpora |

Reproduce any of them: the corpus row above plus `scripts/extract_emotion_vectors.py`.

**Blessed vector bundles (post padding-fix).** As of 2026-07-22 the blessed extractions are
the `-postfix` sets: `results/emotion_vectors_postfix`, `results/emotion_vectors_it_postfix`,
`results/self_story_vectors_it_postfix`, and `results/neutral_vectors_it_postfix`, re-extracted
after the padding bug fix (TREE node Q1.H3.E4b). Their published homes are the matching
`-postfix` repos on Hugging Face, each carrying a `LINEAGE.md` that records the fix. The
unsuffixed sets above are retained unchanged as the pre-2026-07-22 record, for the E4b
before/after comparison and for reproducing pre-fix results.

In [3]:
# this cell sizes each blessed -postfix bundle from its manifest (emotions, stories, tokens)
POSTFIX_SETS = [
    "emotion_vectors_postfix",
    "emotion_vectors_it_postfix",
    "self_story_vectors_it_postfix",
    "neutral_vectors_it_postfix",
]
rows = []
for name in POSTFIX_SETS:
    manifest = [json.loads(line) for line in open(fetch(f"{name}/manifest.jsonl"))]
    ok = [entry for entry in manifest if entry["error"] is None]
    rows.append(
        (
            f"results/{name}",
            len({entry["emotion"] for entry in ok}),
            len(ok),
            f"{sum(entry['n_tokens'] for entry in ok):,}",
            len(manifest) - len(ok),
        )
    )
fig = go.Figure(
    go.Table(
        header=dict(
            values=["blessed bundle", "emotions", "stories", "tokens", "errors"], align="left"
        ),
        cells=dict(values=list(zip(*rows)), align="left", height=26),
    )
)
fig.update_layout(
    title="Blessed -postfix vector bundles (Q1.H3.E4b), sized from their extraction manifests",
    height=280,
    margin=dict(t=50, b=10),
)
fig.show()
